In [1]:
import os
import glob
from typing import Dict, Tuple, List

import numpy as np
import rasterio
import os
import glob
import re
import numpy as np
import rasterio
from typing import Optional, Union

def _safe(arr: np.ndarray) -> np.ndarray:
    a = np.asarray(arr, dtype=float).ravel()
    return a

def _nanmean(a): return np.nanmean(a)
def _nanstd(a):  return np.nanstd(a, ddof=1)  # 与样本标准差一致
def _nanmin(a):  return np.nanmin(a)
def _nanmax(a):  return np.nanmax(a)
def _nanpct(a, q): return np.nanpercentile(a, q)

def _linear_trend_slope(a: np.ndarray) -> float:
    a = _safe(a)
    n = a.size
    t = np.arange(n, dtype=float)
    mask = ~np.isnan(a)
    if mask.sum() < 2:
        return np.nan
    t, y = t[mask], a[mask]
    # 斜率 = cov(t,y)/var(t)
    var_t = np.var(t)
    if var_t == 0:
        return np.nan
    return np.cov(t, y, bias=True)[0,1] / var_t

def _autocorr(a: np.ndarray, lag: int) -> float:
    a = _safe(a)
    if a.size <= lag:
        return np.nan
    x = a[:-lag]
    y = a[lag:]
    mask = ~np.isnan(x) & ~np.isnan(y)
    if mask.sum() < 2:
        return np.nan
    x = x[mask]
    y = y[mask]
    x = (x - x.mean())
    y = (y - y.mean())
    denom = np.sqrt((x**2).sum() * (y**2).sum())
    return (x*y).sum() / denom if denom != 0 else np.nan

def _longest_run_below_threshold(a: np.ndarray, thr: float) -> int:
    a = _safe(a)
    best = cur = 0
    for v in a:
        if not np.isnan(v) and v < thr:
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
    return best

def _time_since_last_event(a: np.ndarray, thr: float) -> float:
    a = _safe(a)
    for i in range(a.size-1, -1, -1):
        v = a[i]
        if not np.isnan(v) and v < thr:
            return a.size - 1 - i
    return np.nan

def _seasonality_index(a: np.ndarray, months_in_year: int = 12) -> float:
    a = _safe(a)
    n = a.size
    if months_in_year <= 0 or n < months_in_year or n % months_in_year != 0:
        return np.nan
    k = n // months_in_year
    A = a.reshape(k, months_in_year)
    monthly_means = np.nanmean(A, axis=0)  # 12个
    overall_mean  = np.nanmean(a)
    if np.isnan(overall_mean) or overall_mean == 0:
        return np.nan
    return (np.nanmax(monthly_means) - np.nanmin(monthly_means)) / overall_mean

def _yearly_stats(a: np.ndarray, months_in_year: int = 12) -> Tuple[float, float]:
    a = _safe(a)
    n = a.size
    if months_in_year <= 0 or n < months_in_year or n % months_in_year != 0:
        return (np.nan, np.nan)
    k = n // months_in_year
    A = a.reshape(k, months_in_year)
    yearly_means = np.nanmean(A, axis=1)  # k个
    std_y = np.nanstd(yearly_means, ddof=1) if k > 1 else np.nan
    mean_y = np.nanmean(yearly_means)
    cv_y = std_y / mean_y if (not np.isnan(std_y) and mean_y not in [0, np.nan]) else np.nan
    return (std_y, cv_y)

def _fourier_amp_phase(a: np.ndarray, k: int, months_in_year: int = 12) -> Tuple[float, float]:
    a = _safe(a)
    n = a.size
    t = np.arange(n, dtype=float)
    mask = ~np.isnan(a)
    if mask.sum() < 3:
        return (np.nan, np.nan)
    t = t[mask]
    y = a[mask]
    w = 2.0 * np.pi * k / months_in_year
    C = np.column_stack([np.cos(w*t), np.sin(w*t)])

    coeff, _, _, _ = np.linalg.lstsq(C, y, rcond=None)
    A, B = coeff[0], coeff[1]
    amplitude = np.sqrt(A*A + B*B)
    phase = np.arctan2(B, A)  
    return (amplitude, phase)

def compute_common_ts_features(x: np.ndarray, months_in_year: int = 12) -> Dict[str, float]:

    x = _safe(x)

    mu   = _nanmean(x)
    sd   = _nanstd(x)
    cv   = sd / mu if (not np.isnan(sd) and mu not in [0, np.nan]) else np.nan
    xmin = _nanmin(x)
    xmax = _nanmax(x)
    rng  = xmax - xmin if (not np.isnan(xmax) and not np.isnan(xmin)) else np.nan

    slope = _linear_trend_slope(x)
    ac1   = _autocorr(x, lag=1)
    ac12  = _autocorr(x, lag=12)

    q20 = _nanpct(x, 20)
    q80 = _nanpct(x, 80)
    below20 = np.sum(x < q20) if not np.isnan(q20) else np.nan
    above80 = np.sum(x > q80) if not np.isnan(q80) else np.nan
    T = np.count_nonzero(~np.isnan(x))
    frac_low  = (below20 / T) if (T and not np.isnan(below20)) else np.nan
    frac_high = (above80 / T) if (T and not np.isnan(above80)) else np.nan

    longest_low = _longest_run_below_threshold(x, q20) if not np.isnan(q20) else np.nan
    tsince_low  = _time_since_last_event(x, q20) if not np.isnan(q20) else np.nan

    integral = np.nansum(x)
    sma = np.nansum(x - mu) if not np.isnan(mu) else np.nan

    seas_idx = _seasonality_index(x, months_in_year=months_in_year)
    inter_std, inter_cv = _yearly_stats(x, months_in_year=months_in_year)

    amp1, phi1 = _fourier_amp_phase(x, k=1, months_in_year=months_in_year)
    amp2, phi2 = _fourier_amp_phase(x, k=2, months_in_year=months_in_year)

    return {
        "mean": mu,
        "std": sd,
        "cv": cv,
        "min": xmin,
        "max": xmax,
        "range": rng,
        "trend_slope_per_month": slope,
        "autocorr_lag1": ac1,
        "autocorr_lag12": ac12,
        "q20": q20,
        "q80": q80,
        "frac_below_q20": frac_low,
        "frac_above_q80": frac_high,
        "longest_below_q20": longest_low,
        "time_since_last_q20": tsince_low,
        "integral_sum": integral,
        "sma_anomaly_sum": sma,
        "seasonality_index": seas_idx,
        "interannual_std": inter_std,
        "interannual_cv": inter_cv,
        "fourier1_amplitude": amp1,
        "fourier1_phase_rad": phi1,
        "fourier2_amplitude": amp2,
        "fourier2_phase_rad": phi2,
    }


def build_composite(
    input_dir: str,
    out_path: str,
    pattern: str,
    start_ym: Optional[Union[int, str]] = None,   # 例如 201501
    end_ym: Optional[Union[int, str]] = None,     # 例如 202212
) -> str:

    files = sorted(glob.glob(os.path.join(input_dir, pattern)))
    if len(files) == 0:
        raise RuntimeError(f"在 {input_dir} 中未找到匹配 {pattern} 的文件")

    if start_ym is not None and end_ym is not None:
        start_ym = int(start_ym)
        end_ym = int(end_ym)

        filtered_files = []
        for f in files:
            fname = os.path.basename(f)
            # 尝试从文件名中抓 6 位数字当作 YYYYMM
            m = re.search(r"(\d{6})", fname)
            if not m:
                continue
            ym = int(m.group(1))
            if start_ym <= ym <= end_ym:
                filtered_files.append(f)

        files = sorted(filtered_files)
        if len(files) == 0:
            raise RuntimeError(
                f"在 {input_dir} 中匹配 {pattern} 但在 {start_ym}~{end_ym} 范围内没有文件"
            )

    print(f"找到 {len(files)} 个 TIF，将合并为多波段 TIF：")
    for f in files:
        print("  ", os.path.basename(f))

    with rasterio.open(files[0]) as src0:
        profile = src0.profile.copy()

    profile.update(
        count=len(files),
        dtype="float32",   # 后面会转为 float32 + NaN
        nodata=np.nan
    )

    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    with rasterio.open(out_path, "w", **profile) as dst:
        for i, fname in enumerate(files, start=1):
            with rasterio.open(fname) as src:
                arr = src.read(1).astype("float32")
                # 把原来的 nodata 转成 NaN
                if src.nodata is not None:
                    arr = np.where(arr == src.nodata, np.nan, arr)
                dst.write(arr, i)

    print(f"✅ 合并完成，输出：{out_path}")
    return out_path

def ts_features_from_composite(
    composite_path: str,
    out_dir: str,
    months_in_year: int = 12,
    batch_size: int = 20000
):

    os.makedirs(out_dir, exist_ok=True)

    with rasterio.open(composite_path) as src:
        data = src.read()  # (T, H, W)
        T, H, W = data.shape
        print(f"composite 形状：T={T}, H={H}, W={W}")
       
        if src.nodata is not None:
            data = data.astype("float32")
            data = np.where(data == src.nodata, np.nan, data)

        Npix = H * W
        ts_matrix = data.reshape(T, Npix).T  # (Npix, T)

        dummy = compute_common_ts_features(np.full(T, np.nan), months_in_year=months_in_year)
        feat_names = list(dummy.keys())
        print("将计算特征：", feat_names)

        feat_arrays = {
            name: np.full(Npix, np.nan, dtype="float32")
            for name in feat_names
        }

        for start in range(0, Npix, batch_size):
            end = min(Npix, start + batch_size)
            batch = ts_matrix[start:end, :]  # (B, T)
            B = batch.shape[0]

            if (start // batch_size) % 10 == 0:
                print(f"  处理像元 {start} / {Npix} ...")

            for i in range(B):
                series = batch[i, :]

                if np.all(np.isnan(series)):
                    continue
                feats = compute_common_ts_features(series, months_in_year=months_in_year)
                for name in feat_names:
                    feat_arrays[name][start + i] = np.float32(feats[name] if feats[name] is not None else np.nan)

        profile = src.profile.copy()
        profile.update(
            count=1,
            dtype="float32",
            nodata=np.nan
        )

        for name in feat_names:
            out_tif = os.path.join(out_dir, f"FL_tmean_ts_{name}.tif")
            arr2d = feat_arrays[name].reshape(H, W)
            with rasterio.open(out_tif, "w", **profile) as dst:
                dst.write(arr2d, 1)
            print(f"  ✅ 写出特征 {name}: {out_tif}")

    print("🎉 所有时间序列特征 TIF 生成完毕！")


In [2]:
INPUT_DIR = "/home/ji.song/orange/Jiayi/Grazing_SOC_Feature_TIF_v2/tif_set/con"  

COMPOSITE_TIF = "/home/ji.song/orange/Jiayi/Grazing_SOC_Feature_TIF_v2/tif_set/FL_tmean_stack.tif"

OUT_TS_DIR = "/home/ji.song/orange/Jiayi/Grazing_SOC_Feature_TIF_v2/tif_set/FL_tmean_ts/indicators"

months_in_year = 12 

In [4]:
import os
import glob
from datetime import datetime

all_files = glob.glob(os.path.join(INPUT_DIR, "prism_tmean_us_30s_*.tif"))

selected_files = []
for f in all_files:
    fname = os.path.basename(f)  # prism_tmean_us_30s_201501.tif
    date_str = fname.replace("prism_tmean_us_30s_", "").replace(".tif", "")

    try:
        dt = datetime.strptime(date_str, "%Y%m")
    except:
        continue

    if datetime(2015, 1, 1) <= dt <= datetime(2022, 12, 1):
        selected_files.append(f)

print("筛选后的文件数：", len(selected_files))

composite_path = build_composite(
    input_dir=INPUT_DIR,
    out_path=COMPOSITE_TIF,
    pattern="prism_tmean_us_30s_20*.tif", 
    start_ym=201501,
    end_ym=202212,
)

筛选后的文件数： 96
找到 96 个 TIF，将合并为多波段 TIF：
   prism_tmean_us_30s_201501.tif
   prism_tmean_us_30s_201502.tif
   prism_tmean_us_30s_201503.tif
   prism_tmean_us_30s_201504.tif
   prism_tmean_us_30s_201505.tif
   prism_tmean_us_30s_201506.tif
   prism_tmean_us_30s_201507.tif
   prism_tmean_us_30s_201508.tif
   prism_tmean_us_30s_201509.tif
   prism_tmean_us_30s_201510.tif
   prism_tmean_us_30s_201511.tif
   prism_tmean_us_30s_201512.tif
   prism_tmean_us_30s_201601.tif
   prism_tmean_us_30s_201602.tif
   prism_tmean_us_30s_201603.tif
   prism_tmean_us_30s_201604.tif
   prism_tmean_us_30s_201605.tif
   prism_tmean_us_30s_201606.tif
   prism_tmean_us_30s_201607.tif
   prism_tmean_us_30s_201608.tif
   prism_tmean_us_30s_201609.tif
   prism_tmean_us_30s_201610.tif
   prism_tmean_us_30s_201611.tif
   prism_tmean_us_30s_201612.tif
   prism_tmean_us_30s_201701.tif
   prism_tmean_us_30s_201702.tif
   prism_tmean_us_30s_201703.tif
   prism_tmean_us_30s_201704.tif
   prism_tmean_us_30s_201705.tif
   pri

In [6]:
ts_features_from_composite(
    composite_path=COMPOSITE_TIF,
    out_dir=OUT_TS_DIR,
    months_in_year=months_in_year,
    batch_size=8000  # 可以根据内存情况调节
)

composite 形状：T=96, H=3105, W=7025


/apps/geopython/1.0.1/lib/python3.7/site-packages/ipykernel/__main__.py:22: RuntimeWarning: Mean of empty slice
/apps/geopython/1.0.1/lib/python3.7/site-packages/numpy/lib/nanfunctions.py:1665: RuntimeWarning: Degrees of freedom <= 0 for slice.
  keepdims=keepdims)
/apps/geopython/1.0.1/lib/python3.7/site-packages/ipykernel/__main__.py:24: RuntimeWarning: All-NaN slice encountered
/apps/geopython/1.0.1/lib/python3.7/site-packages/ipykernel/__main__.py:25: RuntimeWarning: All-NaN slice encountered
/apps/geopython/1.0.1/lib/python3.7/site-packages/numpy/lib/nanfunctions.py:1370: RuntimeWarning: All-NaN slice encountered
  overwrite_input=overwrite_input, interpolation=interpolation
/apps/geopython/1.0.1/lib/python3.7/site-packages/ipykernel/__main__.py:93: RuntimeWarning: Mean of empty slice
/apps/geopython/1.0.1/lib/python3.7/site-packages/ipykernel/__main__.py:94: RuntimeWarning: Mean of empty slice
/apps/geopython/1.0.1/lib/python3.7/site-packages/ipykernel/__main__.py:110: RuntimeWar

将计算特征： ['mean', 'std', 'cv', 'min', 'max', 'range', 'trend_slope_per_month', 'autocorr_lag1', 'autocorr_lag12', 'q20', 'q80', 'frac_below_q20', 'frac_above_q80', 'longest_below_q20', 'time_since_last_q20', 'integral_sum', 'sma_anomaly_sum', 'seasonality_index', 'interannual_std', 'interannual_cv', 'fourier1_amplitude', 'fourier1_phase_rad', 'fourier2_amplitude', 'fourier2_phase_rad']
  处理像元 0 / 21812625 ...
  处理像元 80000 / 21812625 ...
  处理像元 160000 / 21812625 ...
  处理像元 240000 / 21812625 ...
  处理像元 320000 / 21812625 ...
  处理像元 400000 / 21812625 ...
  处理像元 480000 / 21812625 ...
  处理像元 560000 / 21812625 ...
  处理像元 640000 / 21812625 ...
  处理像元 720000 / 21812625 ...
  处理像元 800000 / 21812625 ...
  处理像元 880000 / 21812625 ...
  处理像元 960000 / 21812625 ...
  处理像元 1040000 / 21812625 ...
  处理像元 1120000 / 21812625 ...
  处理像元 1200000 / 21812625 ...
  处理像元 1280000 / 21812625 ...
  处理像元 1360000 / 21812625 ...
  处理像元 1440000 / 21812625 ...
  处理像元 1520000 / 21812625 ...
  处理像元 1600000 / 21812625 ...
  

# Fire

In [5]:
import math
from pathlib import Path

import numpy as np
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize
from rasterio.enums import MergeAlg

SHP_IN  = Path("/home/ji.song/orange/Jiayi/Grazing_SOC_Feature_TIF_v2/Fire_Shapefire/fire2005.shp")
OUT_DIR = Path("/home/ji.song/orange/Jiayi/Grazing_SOC_Feature_TIF_v2/Fire_Rasters_30m")  
OUT_DIR.mkdir(parents=True, exist_ok=True)

RADII_M = [250, 750, 1250]       
RES = 30.0                       
OUT_NAME_FMT = "fire_2005_{r}m_30m.tif"  

def is_projected_meters(crs):
    try:
        return (crs is not None) and (not crs.is_geographic)
    except Exception:
        return False

fires = gpd.read_file(SHP_IN)
if "Shape_Area" not in fires.columns:
    raise ValueError("Shapefile 缺少 `Shape_Area` 列。")

print("原始 CRS:", fires.crs)

if is_projected_meters(fires.crs):
    target_crs = fires.crs
else:

    target_crs = "EPSG:3086"  # Florida Albers
    print("将 fire 图层重投影到", target_crs)

fires_p = fires.to_crs(target_crs)[["Shape_Area", "geometry"]].copy()

minx, miny, maxx, maxy = fires_p.total_bounds
max_r = max(RADII_M)

minx -= max_r
miny -= max_r
maxx += max_r
maxy += max_r

width  = math.ceil((maxx - minx) / RES)
height = math.ceil((maxy - miny) / RES)

transform = from_origin(minx, maxy, RES, RES)

print(f"栅格范围：")
print(f"  X: [{minx}, {maxx}]")
print(f"  Y: [{miny}, {maxy}]")
print(f"栅格大小：width={width}, height={height}, res={RES} m")

base_profile = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "float32",
    "crs": target_crs,
    "transform": transform,
    "nodata": np.nan,
    "compress": "lzw",
}

for r in RADII_M:
    print(f"\n=== 处理半径 r = {r} m ===")

    # 1) 对 fire polygon 做 r 米缓冲
    fires_buf = fires_p.copy()
    fires_buf["geometry"] = fires_buf.geometry.buffer(r)

    # 2) 准备 (geometry, value) 列表，值为 Shape_Area
    shapes = [
        (geom, float(area))
        for geom, area in zip(fires_buf.geometry, fires_buf["Shape_Area"])
        if geom is not None and not geom.is_empty
    ]

    print(f"  缓冲后多边形数量：{len(shapes)}")

    out_arr = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0.0,           
        dtype="float32",
        all_touched=False,   
        merge_alg=MergeAlg.add,
    )

    out_profile = base_profile.copy()
    out_path = OUT_DIR / OUT_NAME_FMT.format(r=r)

    with rasterio.open(out_path, "w", **out_profile) as dst:
        dst.write(out_arr, 1)

    print(f"  ✅ 已输出: {out_path}")

print("\n🎉 所有 fire 特征栅格生成完毕！")


原始 CRS: PROJCS["IMAGINE_GeoTIFF_Support_ERDAS_IMAGINE_2018_16_5_0_596_Projection_Albers_Conical",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["Meters",1],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
栅格范围：
  X: [733165.0, 1597595.0]
  Y: [350485.0, 1215335.0]
栅格大小：width=28815, height=28829, res=30.0 m

=== 处理半径 r = 250 m ===
  缓冲后多边形数量：426503
  ✅ 已输出: /home/ji.song/orange/Jiayi/Grazing_SOC_Feature_TIF_v2/Fire_Rasters_30m/fire_2005_250m_30m.tif

=== 处理半径 r = 750 m ===
  缓冲后多边形数量：426503
  ✅ 已输出: /home/ji.song/orange/Jiayi/Grazing_SOC_Feature_TIF_v2/Fire_Rasters_30m/fire_2005_750m_30m.